<a href="https://colab.research.google.com/github/Haross/sql_notebook/blob/main/assignments/Pyspark/Assignment_Pyspark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("Spark SQL Assignment") \
    .getOrCreate()

In [9]:
%%capture
!wget -O "nested_pyspark_assignment_dataset.json" "https://raw.githubusercontent.com/Haross/sql_notebook/main/assignments/Pyspark/nested_pyspark_assignment_dataset.json"


In [2]:
# @title Schema Set Up
from pyspark.sql.types import *

schema = StructType([
    StructField("company_id", StringType()),
    StructField("company_name", StringType()),
    StructField("snapshot_date", StringType()),

    StructField("business_units", ArrayType(
        StructType([
            StructField("business_unit_id", StringType()),
            StructField("business_unit_name", StringType()),

            # ---------- FULFILLMENT BRANCH ----------
            StructField("fulfillment_branch", StructType([
                StructField("regions_by_code", MapType(
                    StringType(),
                    StructType([
                        StructField("region_name", StringType()),
                        StructField("regional_manager", StringType()),

                        StructField("warehouses", ArrayType(
                            StructType([
                                StructField("warehouse_id", StringType()),
                                StructField("warehouse_name", StringType()),
                                StructField("city", StringType()),

                                StructField("warehouse_attributes", StructType([
                                    StructField("capacity_tier", StringType()),
                                    StructField("temperature_controlled", StringType())
                                ])),

                                StructField("zones", ArrayType(
                                    StructType([
                                        StructField("zone_id", StringType()),
                                        StructField("zone_type", StringType()),
                                        StructField("temperature_readings", ArrayType(
                                            StructType([
                                                StructField("reading_time", StringType()),
                                                StructField("temperature_celsius", DoubleType()),
                                                StructField("humidity_percent", DoubleType())
                                            ])
                                        ))
                                    ])
                                )),

                                StructField("shipments", ArrayType(
                                    StructType([
                                        StructField("shipment_id", StringType()),
                                        StructField("carrier", StringType()),
                                        StructField("shipment_status", StringType()),

                                        StructField("tracking_events", ArrayType(
                                            StructType([
                                                StructField("event_time", StringType()),
                                                StructField("event_type", StringType()),
                                                StructField("location", StringType()),
                                                StructField("scan_metadata", StructType([
                                                    StructField("scanner_id", StringType()),
                                                    StructField("device", StructType([
                                                        StructField("device_id", StringType()),
                                                        StructField("device_model", StringType()),
                                                        StructField("battery_level", IntegerType())
                                                    ]))
                                                ]))
                                            ])
                                        )),

                                        StructField("packages", ArrayType(
                                            StructType([
                                                StructField("package_id", StringType()),
                                                StructField("package_type", StringType()),
                                                StructField("weight_kg", DoubleType()),

                                                StructField("items", ArrayType(
                                                    StructType([
                                                        StructField("item_id", StringType()),
                                                        StructField("sku", StringType()),
                                                        StructField("product_name", StringType()),
                                                        StructField("quantity", IntegerType()),
                                                        StructField("unit_price", DoubleType()),

                                                        # map instead of struct
                                                        StructField("item_attributes", MapType(
                                                            StringType(),
                                                            StringType()
                                                        )),

                                                        StructField("quality_checks", ArrayType(
                                                            StructType([
                                                                StructField("check_id", StringType()),
                                                                StructField("check_type", StringType()),
                                                                StructField("passed", BooleanType()),
                                                                StructField("inspector_id", StringType()),

                                                                StructField("defects", ArrayType(
                                                                    StructType([
                                                                        StructField("defect_id", StringType()),
                                                                        StructField("defect_type", StringType()),
                                                                        StructField("severity", StringType()),

                                                                        StructField("evidence", ArrayType(
                                                                            StructType([
                                                                                StructField("evidence_id", StringType()),
                                                                                StructField("evidence_type", StringType()),
                                                                                StructField("file_url", StringType())
                                                                            ])
                                                                        ))
                                                                    ])
                                                                ))
                                                            ])
                                                        ))
                                                    ])
                                                ))
                                            ])
                                        ))
                                    ])
                                ))
                            ])
                        ))
                    ])
                ))
            ])),

            # ---------- FINANCE BRANCH ----------
            StructField("finance_branch", StructType([
                StructField("billing_accounts_by_id", MapType(
                    StringType(),
                    StructType([
                        StructField("account_region_code", StringType()),
                        StructField("customer_segment", StringType()),

                        StructField("invoices", ArrayType(
                            StructType([
                                StructField("invoice_id", StringType()),
                                StructField("shipment_id", StringType()),
                                StructField("invoice_status", StringType()),

                                StructField("charges", ArrayType(
                                    StructType([
                                        StructField("charge_id", StringType()),
                                        StructField("charge_type", StringType()),
                                        StructField("amount", DoubleType()),
                                        StructField("currency", StringType()),

                                        StructField("tax_breakdown", MapType(
                                            StringType(),
                                            StructType([
                                                StructField("tax_rate", DoubleType()),
                                                StructField("tax_amount", DoubleType())
                                            ])
                                        ))
                                    ])
                                ))
                            ])
                        ))
                    ])
                ))
            ]))
        ])
    ))
])

In [10]:
df_json = spark.read.schema(schema).json("/content/nested_pyspark_assignment_dataset.json")
df_json.show(truncate=False)

+----------+-----------------------+-------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [11]:
df_json.printSchema()

root
 |-- company_id: string (nullable = true)
 |-- company_name: string (nullable = true)
 |-- snapshot_date: string (nullable = true)
 |-- business_units: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- business_unit_id: string (nullable = true)
 |    |    |-- business_unit_name: string (nullable = true)
 |    |    |-- fulfillment_branch: struct (nullable = true)
 |    |    |    |-- regions_by_code: map (nullable = true)
 |    |    |    |    |-- key: string
 |    |    |    |    |-- value: struct (valueContainsNull = true)
 |    |    |    |    |    |-- region_name: string (nullable = true)
 |    |    |    |    |    |-- regional_manager: string (nullable = true)
 |    |    |    |    |    |-- warehouses: array (nullable = true)
 |    |    |    |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |    |    |    |-- warehouse_id: string (nullable = true)
 |    |    |    |    |    |    |    |-- warehouse_name: string (nullable

In [13]:
df_json.createOrReplaceTempView("raw_company")